# Diabetes Prediction using Machine Learning
### Assessment 1 — ML Mini Project

**Problem Statement:** Diabetes is a chronic disease affecting millions worldwide. Early prediction based on
simple diagnostic measurements (glucose level, BMI, age, etc.) can help flag at-risk patients for further
clinical testing. This notebook builds and compares several classification models to predict whether a
patient has diabetes (`Outcome` = 1) based on the **Pima Indians Diabetes Dataset**.

**Why it matters:** Early screening is cheap and non-invasive compared to full diagnostic testing, and a
reliable ML model can be used as a triage tool in low-resource clinical settings.

**Dataset:** 768 patient records, 8 numeric features + binary target (`Outcome`).
Source: Pima Indians Diabetes Database (National Institute of Diabetes and Digestive and Kidney Diseases).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report,
                              RocCurveDisplay)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


## 2. Exploratory Data Analysis (EDA)

Note: several columns (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`) contain
biologically-impossible zero values in the raw data — these represent missing values, not true
measurements of zero, and are handled during preprocessing (Section 3).

In [ ]:
# Target class balance
plt.figure()
ax = sns.countplot(x='Outcome', data=df, palette='viridis')
plt.title('Class Distribution: Diabetes Outcome')
plt.xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.savefig('../notebooks/eda_class_balance.png', dpi=100, bbox_inches='tight')
plt.show()
print(df['Outcome'].value_counts(normalize=True))


In [ ]:
# Distribution of each feature
df.drop('Outcome', axis=1).hist(bins=20, figsize=(14, 10))
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.savefig('../notebooks/eda_histograms.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.savefig('../notebooks/eda_correlation.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Boxplots to check outliers per feature, split by outcome
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), df.drop('Outcome', axis=1).columns):
    sns.boxplot(x='Outcome', y=col, data=df, ax=ax, palette='Set2')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('../notebooks/eda_boxplots.png', dpi=100, bbox_inches='tight')
plt.show()


**EDA takeaways:**
- The dataset is mildly imbalanced (~65% no-diabetes / ~35% diabetes in the original Pima data).
- `Glucose`, `BMI`, and `Age` show the strongest visual separation between the two classes.
- `Insulin` and `SkinThickness` have the most zero/missing values and the widest spread — worth
  paying attention to during preprocessing and model selection.

## 3. Data Preprocessing

In [ ]:
# Treat biologically-invalid zeros as missing values, then impute with the median
cols_with_invalid_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("Zero-value counts before cleaning:")
print((df[cols_with_invalid_zero] == 0).sum())

df_clean = df.copy()
for c in cols_with_invalid_zero:
    df_clean[c] = df_clean[c].replace(0, np.nan)
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())

print("\nMissing values after imputation:")
print(df_clean.isnull().sum())


In [ ]:
X = df_clean.drop('Outcome', axis=1)
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Handling class imbalance with SMOTE
Diabetes datasets are typically imbalanced. We apply SMOTE (Synthetic Minority Over-sampling
Technique) on the **training set only** to avoid data leakage into the test set.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE :", pd.Series(y_train_res).value_counts().to_dict())


## 4. Model Implementation
We train and compare five classification algorithms.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_test_scaled)
    proba = model.predict_proba(X_test_scaled)[:, 1]
    fitted_models[name] = model
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
results_df


## 5. Model Evaluation

In [ ]:
# Bar chart comparing models across metrics
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', figsize=(12, 6))
plt.title('Model Comparison Across Metrics')
plt.ylabel('Score')
plt.xticks(rotation=30)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../notebooks/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = fitted_models[best_model_name]
print("Best performing model:", best_model_name)

preds = best_model.predict(X_test_scaled)
print(classification_report(y_test, preds))

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('../notebooks/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# ROC curves for all models
plt.figure(figsize=(8, 6))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test, ax=plt.gca(), name=name)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC Curves — All Models')
plt.savefig('../notebooks/roc_curves.png', dpi=100, bbox_inches='tight')
plt.show()


## 6. Model Improvement (Hyperparameter Tuning)
We tune the best-performing model family using `GridSearchCV` with 5-fold cross-validation,
optimizing for F1-score (a better metric than accuracy on imbalanced medical data, since it
balances false positives and false negatives).

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid.fit(X_train_res, y_train_res)

print("Best parameters:", grid.best_params_)
print("Best CV F1-score:", grid.best_score_)

tuned_model = grid.best_estimator_
tuned_preds = tuned_model.predict(X_test_scaled)
tuned_proba = tuned_model.predict_proba(X_test_scaled)[:, 1]

print("\nTuned model performance on test set:")
print("Accuracy :", accuracy_score(y_test, tuned_preds))
print("Precision:", precision_score(y_test, tuned_preds))
print("Recall   :", recall_score(y_test, tuned_preds))
print("F1-Score :", f1_score(y_test, tuned_preds))
print("ROC-AUC  :", roc_auc_score(y_test, tuned_proba))


In [ ]:
# Feature importance from the tuned model
importances = pd.Series(tuned_model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette='mako')
plt.title('Feature Importance — Tuned Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../notebooks/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()


## 7. Save Final Model
We persist the tuned model and the scaler so the Streamlit app (in `../app/app.py`) can load
them directly for inference, without retraining.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(tuned_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Also save which columns are used and in what order, for the app's input form
joblib.dump(list(X.columns), '../models/feature_columns.pkl')

print("Model, scaler, and feature list saved to ../models/")


## 8. Conclusion

- Compared six classification algorithms (Logistic Regression, Decision Tree, Random Forest, KNN, SVM, XGBoost).
- Handled missing/invalid values, scaled features, and corrected class imbalance with SMOTE.
- Tuned the best model family via `GridSearchCV`, improving F1-score over the untuned baseline.
- `Glucose`, `BMI`, and `Age` were consistently the most important predictors — consistent with clinical
  understanding of diabetes risk factors.
- The final model and scaler are exported for use in the deployed Streamlit application.

**Possible future improvements:** collect a larger/more diverse dataset, try stacking/ensembling multiple
models, add SHAP explainability for clinical interpretability.